In [1]:
import json
import joblib
import torch
import numpy as np
from transformers import AutoTokenizer, T5EncoderModel

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
encoder= T5EncoderModel.from_pretrained("/home/sasha/Python/VKR/pytorch_bert/model_weights/encoder")

Loading weights: 100%|██████████| 219/219 [00:00<00:00, 1031.28it/s]


In [3]:
tokenizer = AutoTokenizer.from_pretrained("/home/sasha/Python/VKR/pytorch_bert/model_weights/tokenizer")

In [4]:
encoder.eval()

T5EncoderModel(
  (shared): Embedding(93651, 1536)
  (encoder): T5Stack(
    (embed_tokens): Embedding(93651, 1536)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1536, out_features=1536, bias=False)
              (k): Linear(in_features=1536, out_features=1536, bias=False)
              (v): Linear(in_features=1536, out_features=1536, bias=False)
              (o): Linear(in_features=1536, out_features=1536, bias=False)
              (relative_attention_bias): Embedding(32, 24)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1536, out_features=4096, bias=False)
              (wi_1): Linear(in_features=1536, out_features=4096, bias=False)
              (wo):

In [5]:
classifier = joblib.load("/home/sasha/Python/VKR/pytorch_bert/model_weights/classifier.joblib")

In [6]:
tokens=tokenizer(["курьер опоздал","маленький асортимент"],return_tensors="pt", padding=True, truncation=True, max_length=512)

In [7]:
outputs = encoder(**tokens)
embeddings = outputs.last_hidden_state.mean(dim=1).detach().numpy()

In [8]:
embeddings.shape

(2, 1536)

In [9]:
pred_p=classifier.predict_proba(embeddings)

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [10]:
pred_p

[array([[9.99999981e-01, 1.85341882e-08],
        [1.33949126e-01, 8.66050874e-01]]),
 array([[9.99999966e-01, 3.38770261e-08],
        [9.99999881e-01, 1.19053889e-07]]),
 array([[4.60947651e-04, 9.99539052e-01],
        [9.99999032e-01, 9.67627258e-07]]),
 array([[9.99999934e-01, 6.60533169e-08],
        [9.99994613e-01, 5.38713273e-06]]),
 array([[9.99999310e-01, 6.90161968e-07],
        [9.99999107e-01, 8.92690063e-07]]),
 array([[9.99999757e-01, 2.42865510e-07],
        [9.99999969e-01, 3.09330314e-08]]),
 array([[9.99999842e-01, 1.58144496e-07],
        [9.99996433e-01, 3.56735487e-06]]),
 array([[9.99999055e-01, 9.45319128e-07],
        [9.99998231e-01, 1.76883696e-06]])]

In [11]:
pred_pembeddings = outputs.last_hidden_state.mean(dim=1).detach().numpy()

In [12]:
pred_pembeddings

array([[-0.00565326, -0.01478718, -0.00120202, ..., -0.03157436,
        -0.01113608,  0.01231425],
       [-0.0090427 , -0.01191865, -0.01276582, ..., -0.0157279 ,
         0.0070105 ,  0.00992081]], shape=(2, 1536), dtype=float32)

In [14]:
with open('/home/sasha/Python/VKR/pytorch_bert/model_weights/thresholds.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

In [15]:
predict=[]
for i in range(len(pred_p)):
    if pred_p[i][0][1]>=data[i]:
        predict.append(1)
    else:
        predict.append(0)

In [16]:
predict

[0, 0, 1, 0, 0, 0, 0, 0]

In [17]:
pred_p

[array([[9.99999981e-01, 1.85341882e-08],
        [1.33949126e-01, 8.66050874e-01]]),
 array([[9.99999966e-01, 3.38770261e-08],
        [9.99999881e-01, 1.19053889e-07]]),
 array([[4.60947651e-04, 9.99539052e-01],
        [9.99999032e-01, 9.67627258e-07]]),
 array([[9.99999934e-01, 6.60533169e-08],
        [9.99994613e-01, 5.38713273e-06]]),
 array([[9.99999310e-01, 6.90161968e-07],
        [9.99999107e-01, 8.92690063e-07]]),
 array([[9.99999757e-01, 2.42865510e-07],
        [9.99999969e-01, 3.09330314e-08]]),
 array([[9.99999842e-01, 1.58144496e-07],
        [9.99996433e-01, 3.56735487e-06]]),
 array([[9.99999055e-01, 9.45319128e-07],
        [9.99998231e-01, 1.76883696e-06]])]

In [18]:
import pandas as pd

In [19]:
df=pd.read_csv("/home/sasha/Python/VKR/pytorch_bert/new_ds.csv")#names=["text","ASSORTMENT","PROMOTIONS","DELIVERY","PRICE","PRODUCTS_QUALITY","SUPPORT","CATALOG_NAVIGATION","PAYMENT"])

In [20]:
texts=df["text"]

In [23]:
import torch
import numpy as np
import json
import joblib
from transformers import AutoTokenizer, T5EncoderModel

class ClassifierFrida:
    def __init__(self, tokenizer_path, encoder_path, model_path, threshold_path):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.encoder = T5EncoderModel.from_pretrained(encoder_path)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.encoder.to(self.device)
        self.encoder.eval()
        
        self.model = joblib.load(model_path)
        self.threshold_path = threshold_path
        self.load_thresholds()

    def load_thresholds(self):
        """Загружает пороги из файла."""
        with open(self.threshold_path, "r") as f:
            self.thresholds = np.array(json.load(f))

    def get_embeddings(self, texts):
        """Получает эмбеддинги (без батчей, вспомогательный метод)."""
        tokens = self.tokenizer(texts, return_tensors="pt", padding=True, 
                                 truncation=True, max_length=512).to(self.device)
        with torch.no_grad():
            outputs = self.encoder(**tokens)
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().detach().numpy()
        return embeddings

    def get_probs(self, texts, batch_size=32):
        """
        Прогоняет тексты через энкодер и классификатор.
        Возвращает матрицу вероятностей (SAMPLES, CLASSES).
        """
        all_probs = []
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            embeddings = self.get_embeddings(batch_texts)
            
            pred_p = self.model.predict_proba(embeddings)
            
            if isinstance(pred_p, list):
                batch_probs = np.column_stack([p[:, 1] for p in pred_p])
            else:
                batch_probs = pred_p[:, 1] if pred_p.ndim > 1 else pred_p
            
            all_probs.append(batch_probs)
            print(f"Эмбеддинги и вероятности: {min(i + batch_size, len(texts))}/{len(texts)}")
        return np.vstack(all_probs)

    def apply_threshold(self, probs, custom_thresholds=None):
        """
        Применяет пороги к готовым вероятностям. 
        Можно передать новые пороги без перезагрузки класса.
        """
        thr = np.array(custom_thresholds) if custom_thresholds is not None else self.thresholds
        return (probs >= thr).astype(int).tolist()

    def predict(self, texts, batch_size=32):
        """Стандартный полный цикл предсказания."""
        probs = self.get_probs(texts, batch_size)
        return self.apply_threshold(probs)

In [24]:
texts[0]

'Маленький выбор товаров, хотелось бы ассортимент больше, а так вроде бы все хорошо'

In [25]:
model_path_e = "/home/sasha/Python/VKR/pytorch_bert/model_weights/encoder/"
model_path_t = "/home/sasha/Python/VKR/pytorch_bert/model_weights/tokenizer/"
model = "/home/sasha/Python/VKR/pytorch_bert/model_weights/classifier.joblib"
threshold = "/home/sasha/Python/VKR/pytorch_bert/model_weights/thresholds.json"


In [26]:
num_df=df.drop(columns=["text"]).to_numpy()


In [27]:
# 1. Инициализация
pipeline = ClassifierFrida(model_path_t, model_path_e, model, threshold)

# 2. Получаем вероятности ОДИН раз (это долго)
probs = pipeline.get_probs(texts.to_list(), batch_size=16)



Loading weights: 100%|██████████| 219/219 [00:00<00:00, 14411.15it/s]
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/

Эмбеддинги и вероятности: 16/2282


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

Эмбеддинги и вероятности: 32/2282


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

...

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

Эмбеддинги и вероятности: 2240/2282


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

Эмбеддинги и вероятности: 2256/2282
Эмбеддинги и вероятности: 2272/2282
Эмбеддинги и вероятности: 2282/2282


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [28]:
# 3. Тестируем разные пороги без повторного запуска GPU
# Например, стандартные из файла:
results_default = pipeline.apply_threshold(probs)

# Или пробуем завысить пороги (чтобы исправить ваш низкий Precision):
# Допустим, ставим везде 0.8
new_thr = [0.0003, 0.0000254595, 0.0010073387, 0.0002927171, 0.00014049791, 0.00003041178, 0.000165701, 0.0000237321]
results_high = pipeline.apply_threshold(probs, custom_thresholds=new_thr)

# Теперь можно запустить отчет для результатов с новыми порогами
from sklearn.metrics import classification_report
print(classification_report(num_df, results_high))
from sklearn.metrics import hamming_loss

# num_df: true labels (binary matrix)
# pred: predicted labels (binary matrix)
loss = hamming_loss(num_df, results_high)
print(f"Hamming Loss: {loss:.4f}")

              precision    recall  f1-score   support

           0       0.77      0.73      0.75       257
           1       0.49      0.59      0.53        95
           2       0.90      0.88      0.89      1234
           3       0.76      0.79      0.78       401
           4       0.72      0.80      0.76       433
           5       0.57      0.67      0.62       263
           6       0.39      0.45      0.42       137
           7       0.35      0.42      0.38        38

   micro avg       0.75      0.79      0.77      2858
   macro avg       0.62      0.67      0.64      2858
weighted avg       0.76      0.79      0.77      2858
 samples avg       0.65      0.67      0.64      2858

Hamming Loss: 0.0737


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, m

In [ ]:
import optuna
import numpy as np
from sklearn.metrics import hamming_loss

def optimize_hamming_loss(y_probs, y_true, n_trials=200):
    # Превращаем в numpy
    y_probs = np.array(y_probs)
    y_true = np.array(y_true)
    n_classes = y_probs.shape[1]

    def objective(trial):
        # Подбираем пороги для каждого класса
        # Диапазон 0.01 - 0.99
        thresholds = [trial.suggest_float(f"t{i}", 0.01, 0.99) for i in range(n_classes)]
        
        # Применяем пороги
        y_pred = (y_probs >= np.array(thresholds)).astype(int)
        
        # Считаем Hamming Loss (его мы МИНИМИЗИРУЕМ)
        loss = hamming_loss(y_true, y_pred)
        return loss

    # direction="minimize", так как Hamming Loss чем меньше, тем лучше
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    print(f"Минимальный достигнутый Hamming Loss: {study.best_value:.4f}")
    
    best_thresholds = [study.best_params[f"t{i}"] for i in range(n_classes)]
    return best_thresholds

# --- ЗАПУСК ---
# Используем вероятности (probs), которые вы получили ранее через pipeline.get_probs()
best_th_hamming = optimize_hamming_loss(probs, num_df, n_trials=300)

# Применяем лучшие пороги
final_preds = pipeline.apply_threshold(probs, custom_thresholds=best_th_hamming)

# Проверяем метрики
from sklearn.metrics import classification_report, hamming_loss
print("Hamming Loss:", hamming_loss(num_df, final_preds))
print(classification_report(num_df, final_preds, zero_division=0))

[I 2026-04-12 23:28:47,588] A new study created in memory with name: no-name-f4ac9e6d-4dc4-4415-b576-3378a694560e
[I 2026-04-12 23:28:47,591] Trial 0 finished with value: 0.1297655565293602 and parameters: {'t0': 0.10121679068922039, 't1': 0.603836072518334, 't2': 0.8282492313858518, 't3': 0.9170111490856904, 't4': 0.720841171068102, 't5': 0.19662431294735103, 't6': 0.6415037913567656, 't7': 0.9490379394800756}. Best is trial 0 with value: 0.1297655565293602.
[I 2026-04-12 23:28:47,592] Trial 1 finished with value: 0.11431858019281332 and parameters: {'t0': 0.6499104875364561, 't1': 0.07317190701425572, 't2': 0.15212317822853264, 't3': 0.41003733629385397, 't4': 0.8779390117202148, 't5': 0.9698726148360776, 't6': 0.1967809484795934, 't7': 0.6171466202311512}. Best is trial 1 with value: 0.11431858019281332.
[I 2026-04-12 23:28:47,594] Trial 2 finished with value: 0.11273006134969325 and parameters: {'t0': 0.4094307585082743, 't1': 0.3805584445380532, 't2': 0.13028720737900046, 't3': 0.

Минимальный достигнутый Hamming Loss: 0.0884
Hamming Loss: 0.08840929009640666
              precision    recall  f1-score   support

           0       0.91      0.35      0.50       257
           1       1.00      0.06      0.12        95
           2       0.96      0.74      0.84      1234
           3       0.94      0.45      0.61       401
           4       0.99      0.24      0.38       433
           5       1.00      0.02      0.03       263
           6       1.00      0.01      0.01       137
           7       0.00      0.00      0.00        38

   micro avg       0.96      0.45      0.62      2858
   macro avg       0.85      0.23      0.31      2858
weighted avg       0.95      0.45      0.56      2858
 samples avg       0.52      0.43      0.46      2858



In [ ]:
np.sum(final_preds)

np.int64(1356)

In [ ]:
np.sum(results_default)

np.int64(690)

In [ ]:
np.sum(num_df)

np.int64(2858)

In [ ]:
pred=pipeline.predict(texts.to_list())

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

Обработано: 128/2282


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

Обработано: 256/2282


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

...

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

Обработано: 2048/2282


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

Обработано: 2176/2282
Обработано: 2282/2282


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [ ]:
num_df=df.drop(columns=["text"]).to_numpy()


In [ ]:
np.sum(pred)

np.int64(852)

In [ ]:
np.sum(num_df)

np.int64(2858)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(pred,num_df))

              precision    recall  f1-score   support

           0       0.17      0.98      0.29        44
           1       0.08      0.89      0.15         9
           2       0.42      1.00      0.59       519
           3       0.29      0.96      0.45       122
           4       0.21      0.99      0.35        94
           5       0.10      0.96      0.19        28
           6       0.18      0.67      0.28        36
           7       0.00      0.00      0.00         0

   micro avg       0.29      0.97      0.45       852
   macro avg       0.18      0.81      0.29       852
weighted avg       0.34      0.97      0.50       852
 samples avg       0.28      0.34      0.30       852



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{met

In [ ]:
import optuna
import numpy as np
from sklearn.metrics import f1_score

def optimize_with_optuna(y_probs, y_true, n_trials=100):
    # Превращаем в numpy для скорости
    y_probs = np.array(y_probs)
    y_true = np.array(y_true)
    n_classes = y_probs.shape[1]

    def objective(trial):
        # --- ВОТ ЗДЕСЬ ЗАДАЕТСЯ ДИАПАЗОН ---
        # Мы создаем список порогов, по одному для каждого класса
        thresholds = []
        for i in range(n_classes):
            # Optuna предложит значение от 0.01 до 0.99 для каждого класса t0, t1, t2...
            t = trial.suggest_float(f"t{i}", 0.001, 0.99)
            thresholds.append(t)
        
        thresholds = np.array(thresholds)
        
        # Применяем предложенные пороги к вероятностям
        y_pred = (y_probs >= thresholds).astype(int)
        
        # Считаем метрику, которую хотим максимизировать
        score = f1_score(y_true, y_pred, average='macro', zero_division=0)
        return score

    # Создаем процесс поиска
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    print(f"Лучший результат F1: {study.best_value:.4f}")
    
    # Собираем лучшие найденные пороги в список
    best_thresholds = [study.best_params[f"t{i}"] for i in range(n_classes)]
    return best_thresholds

In [ ]:

print("Вероятности собраны. Запускаю Optuna...")
best_th = optimize_with_optuna(pred, num_df, n_trials=1000)

# --- Сохранение ---
with open("/home/sasha/Python/VKR/pytorch_bert/model_weights/thresholds.json", "w") as f:
    import json
    json.dump(best_th, f)

[I 2026-04-12 22:35:56,019] A new study created in memory with name: no-name-5eaf2369-7456-48da-b6f0-3ed626b62665
[I 2026-04-12 22:35:56,023] Trial 0 finished with value: 0.3809991030709555 and parameters: {'t0': 0.8976475927428503, 't1': 0.5735065012730474, 't2': 0.9228899904327799, 't3': 0.9736598469774365, 't4': 0.2028633675976571, 't5': 0.28982196468740323, 't6': 0.7563879906156223, 't7': 0.9203247225298717}. Best is trial 0 with value: 0.3809991030709555.
[I 2026-04-12 22:35:56,026] Trial 1 finished with value: 0.3809991030709555 and parameters: {'t0': 0.302744264696891, 't1': 0.3790242922892021, 't2': 0.837303609937205, 't3': 0.2707110682188226, 't4': 0.16149351358035297, 't5': 0.0704986937760009, 't6': 0.18308726615083662, 't7': 0.2645375608852886}. Best is trial 0 with value: 0.3809991030709555.
[I 2026-04-12 22:35:56,029] Trial 2 finished with value: 0.3809991030709555 and parameters: {'t0': 0.6241316067387559, 't1': 0.7228597623288866, 't2': 0.49523669919659546, 't3': 0.69266

Вероятности собраны. Запускаю Optuna...


[I 2026-04-12 22:35:56,223] Trial 32 finished with value: 0.3809991030709555 and parameters: {'t0': 0.026388439125458407, 't1': 0.4017081803497269, 't2': 0.29184231160028995, 't3': 0.23339576323800837, 't4': 0.9727323550260027, 't5': 0.1337789345095453, 't6': 0.5695422448067993, 't7': 0.7634985943235251}. Best is trial 0 with value: 0.3809991030709555.
[I 2026-04-12 22:35:56,231] Trial 33 finished with value: 0.3809991030709555 and parameters: {'t0': 0.43389283016340646, 't1': 0.5951135563545332, 't2': 0.3977159289199228, 't3': 0.12562027900166567, 't4': 0.4599430287709983, 't5': 0.057699608644918784, 't6': 0.7732563936959835, 't7': 0.9865012502338568}. Best is trial 0 with value: 0.3809991030709555.
[I 2026-04-12 22:35:56,239] Trial 34 finished with value: 0.3809991030709555 and parameters: {'t0': 0.31684394095965407, 't1': 0.33927857398266603, 't2': 0.18698687321530222, 't3': 0.07997841492083939, 't4': 0.7415093711878438, 't5': 0.5604279364309616, 't6': 0.2081876904087196, 't7': 0.89

Лучший результат F1: 0.3810


[0.2, 0.0144595, 0.28073387, 0.04927171, 0.14049791, 0.01041178, 0.00265701, 0.00237321]